In [1]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--force-reinstall", "--no-cache-dir",
    "torch==2.7.1",
    "torchvision==0.22.1",
    "torchaudio==2.7.1",
    "transformers==4.52.4",
    "accelerate==1.7.0",
    "datasets==3.6.0",
    "evaluate==0.4.3",
    "tokenizers==0.21.1"
])

0

In [2]:
import sys
import transformers
import accelerate
import datasets
import torch

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("Datasets:", datasets.__version__)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Transformers: 4.52.4
Accelerate: 1.7.0
Datasets: 3.6.0
Torch: 2.7.1+cu126
CUDA available: True


In [ ]:
# =========================================================
# 1. IMPORTS AND DRIVE SETUP
# =========================================================
import os
import csv
import time
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from torch import nn
from datasets import load_dataset, Dataset

from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    TrainerCallback,
    EarlyStoppingCallback
)

from sklearn.metrics import f1_score, classification_report
from scipy.stats import pearsonr
from google.colab import drive



drive.mount("/content/drive")

os.makedirs("/content/drive/MyDrive", exist_ok=True)

LOG_FILE = "/content/drive/MyDrive/DistilBERT_SingleStep_log.csv"

TEST_EPOCH_SENTENCE_LOG = (
    "/content/drive/MyDrive/DistilBERT_SingleStep_Test_Sentence_Log_Every_Epoch.csv"
)

with open(LOG_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)

    writer.writerow(["model", "distilbert-base-uncased"])
    writer.writerow(["learning_rate", 2e-5])
    writer.writerow(["train_batch_size", 16])
    writer.writerow(["eval_batch_size", 16])
    writer.writerow(["epochs", 10])
    writer.writerow(["sentence_log", "test sentences logged after every epoch"])
    writer.writerow([])


# =========================================================
# 2. SEED SETUP
# =========================================================
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)




# =========================================================
# 3. LOAD BRIGHTER DATASET
# =========================================================
print("\nLoading BRIGHTER dataset from Hugging Face...")

CACHE_DIR = "/content/drive/MyDrive/hf_datasets_cache"

train_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="train",
    cache_dir=CACHE_DIR
)

val_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="dev",
    cache_dir=CACHE_DIR
)

test_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="test",
    cache_dir=CACHE_DIR
)

train_df = train_data.to_pandas()
val_df = val_data.to_pandas()
test_df = test_data.to_pandas()

print("\nOriginal split sizes:")
print("Train:", len(train_df))
print("Dev  :", len(val_df))
print("Test :", len(test_df))

print("\nSample data:")
print(train_df.head())


# =========================================================
# 4. LABEL CONFIGURATION
# =========================================================
EMOTIONS = ["anger", "fear", "joy", "sadness", "surprise"]
LEVELS = [1, 2, 3]

LABELS = [
    f"{emotion}_{level}"
    for emotion in EMOTIONS
    for level in LEVELS
]

NUM_LABELS = len(LABELS)

print("\nLabels:")
print(LABELS)
print("Number of labels:", NUM_LABELS)


# =========================================================
# 5. CONVERT LABELS TO SINGLE-STEP FORMAT
# =========================================================
def convert_single_step_labels(df):
    df = df.copy()

    if "disgust" in df.columns:
        df = df.drop(columns=["disgust"])

    for emotion in EMOTIONS:
        df[f"{emotion}_1"] = (df[emotion] == 1).astype(int)
        df[f"{emotion}_2"] = (df[emotion] == 2).astype(int)
        df[f"{emotion}_3"] = (df[emotion] == 3).astype(int)

    return df


train_single = convert_single_step_labels(train_df)
val_single = convert_single_step_labels(val_df)
test_single = convert_single_step_labels(test_df)

print("\nSingle-step sample:")
print(train_single[["text"] + LABELS].head())


# =========================================================
# 6. COMBINE AND REDISTRIBUTE DATASET: 70 / 20 / 10
# =========================================================
full_df = pd.concat(
    [train_single, val_single, test_single],
    ignore_index=True
)

full_df = full_df[["text"] + LABELS]

full_df = full_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

n = len(full_df)

train_end = int(0.7 * n)
val_end = int(0.9 * n)

train_single = full_df[:train_end].reset_index(drop=True)
val_single = full_df[train_end:val_end].reset_index(drop=True)
test_single = full_df[val_end:].reset_index(drop=True)

print("\nRedistributed split sizes:")
print({
    "train": len(train_single),
    "val": len(val_single),
    "test": len(test_single)
})

# Keep only test texts because we log sentence-wise results only for test set
test_texts = test_single["text"].tolist()


# =========================================================
# 7. CONVERT TO HUGGING FACE DATASET
# =========================================================
train_single = Dataset.from_pandas(train_single, preserve_index=False)
val_single = Dataset.from_pandas(val_single, preserve_index=False)
test_single = Dataset.from_pandas(test_single, preserve_index=False)


# =========================================================
# 8. TOKENIZATION
# =========================================================
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )


train_single = train_single.map(tokenize_function, batched=True)
val_single = val_single.map(tokenize_function, batched=True)
test_single = test_single.map(tokenize_function, batched=True)


# =========================================================
# 9. ADD LABEL VECTOR
# =========================================================
def add_labels(example):
    example["labels"] = [float(example[label]) for label in LABELS]
    return example


train_single = train_single.map(add_labels)
val_single = val_single.map(add_labels)
test_single = test_single.map(add_labels)

train_single.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

val_single.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

test_single.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)


# =========================================================
# 10. METRICS
# =========================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    f1_macro = f1_score(
        labels,
        preds,
        average="macro",
        zero_division=0
    )

    f1_micro = f1_score(
        labels,
        preds,
        average="micro",
        zero_division=0
    )

    pearsons = []

    for i in range(labels.shape[1]):
        if np.std(labels[:, i]) == 0 or np.std(probs[:, i]) == 0:
            pearsons.append(0.0)
        else:
            p, _ = pearsonr(labels[:, i], probs[:, i])
            pearsons.append(0.0 if np.isnan(p) else float(p))

    pearson_mean = float(np.mean(pearsons))

    return {
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "pearson_mean": pearson_mean
    }


# =========================================================
# 11. LISTS FOR PLOTTING AND LOGGING
# =========================================================
epoch_list = []

train_loss_list = []

val_loss_list = []
val_f1_macro_list = []
val_f1_micro_list = []
val_pearson_mean_list = []

test_loss_list = []
test_f1_macro_list = []
test_f1_micro_list = []
test_pearson_mean_list = []


# =========================================================
# 12. HELPER FUNCTIONS FOR SENTENCE-WISE TEST LOGGING
# =========================================================
def clean_number(x):
    """
    Removes unnecessary trailing zeros.
    Example:
    0.0000 -> 0
    0.1500 -> 0.15
    1.0000 -> 1
    """
    x = round(float(x), 4)

    if x == 0:
        return 0

    if x == 1:
        return 1

    return x


def labels_to_text(binary_labels, label_names):
    selected_labels = [
        label_names[i]
        for i, value in enumerate(binary_labels)
        if int(value) == 1
    ]

    if len(selected_labels) == 0:
        return "No Emotion"

    return ", ".join(selected_labels)


# =========================================================
# 13. CALLBACK FOR EPOCH LOGGING + TEST SENTENCE LOGGING
# =========================================================
class SaveEpochResultsCallback(TrainerCallback):
    def __init__(
        self,
        file_path,
        test_dataset,
        test_texts,
        sentence_log_path
    ):
        self.file_path = file_path
        self.test_dataset = test_dataset
        self.test_texts = test_texts
        self.sentence_log_path = sentence_log_path

        self.trainer_ref = None
        self.current_train_loss = None
        self._inside_eval = False

        # Create / reset sentence-wise test log file
        with open(self.sentence_log_path, "w", newline="", encoding="utf-8-sig") as f:
            writer = csv.writer(f)

            header = [
                "epoch",
                "sentence_id",
                "sentence",
                "true_labels",
                "predicted_labels"
            ]

            for label in LABELS:
                header.append(f"prob_{label}")

            writer.writerow(header)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and "loss" in logs and "eval_loss" not in logs:
            self.current_train_loss = float(logs["loss"])

    def write_test_sentence_log_for_epoch(
        self,
        epoch,
        true_labels,
        pred_labels,
        probabilities
    ):
        """
        Writes sentence-wise test predictions for the current epoch.
        """

        with open(self.sentence_log_path, "a", newline="", encoding="utf-8-sig") as f:
            writer = csv.writer(f)

            for i in range(len(self.test_texts)):
                row = [
                    epoch,
                    i,
                    self.test_texts[i],
                    labels_to_text(true_labels[i], LABELS),
                    labels_to_text(pred_labels[i], LABELS)
                ]

                for j in range(len(LABELS)):
                    row.append(clean_number(probabilities[i][j]))

                writer.writerow(row)

        print(f"Sentence-wise test predictions saved for epoch {epoch}.")

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if self._inside_eval:
            return

        if metrics is None:
            return

        self._inside_eval = True

        epoch = int(round(float(metrics.get("epoch", state.epoch))))

        train_loss = (
            self.current_train_loss
            if self.current_train_loss is not None
            else ""
        )

        val_loss = float(metrics.get("eval_loss", 0.0))
        val_f1_macro = float(metrics.get("eval_f1_macro", 0.0))
        val_f1_micro = float(metrics.get("eval_f1_micro", 0.0))
        val_pearson_mean = float(metrics.get("eval_pearson_mean", 0.0))

        test_results = self.trainer_ref.evaluate(
            eval_dataset=self.test_dataset,
            metric_key_prefix="test"
        )

        test_loss = float(test_results.get("test_loss", 0.0))
        test_f1_macro = float(test_results.get("test_f1_macro", 0.0))
        test_f1_micro = float(test_results.get("test_f1_micro", 0.0))
        test_pearson_mean = float(test_results.get("test_pearson_mean", 0.0))

        epoch_list.append(epoch)

        train_loss_list.append(train_loss)

        val_loss_list.append(val_loss)
        val_f1_macro_list.append(val_f1_macro)
        val_f1_micro_list.append(val_f1_micro)
        val_pearson_mean_list.append(val_pearson_mean)

        test_loss_list.append(test_loss)
        test_f1_macro_list.append(test_f1_macro)
        test_f1_micro_list.append(test_f1_micro)
        test_pearson_mean_list.append(test_pearson_mean)

        pred = self.trainer_ref.predict(self.test_dataset)

        logits = pred.predictions
        true_labels = pred.label_ids

        probs = 1 / (1 + np.exp(-logits))
        pred_labels = (probs >= 0.5).astype(int)

        # Sentence-wise test log for this epoch
        self.write_test_sentence_log_for_epoch(
            epoch=epoch,
            true_labels=true_labels,
            pred_labels=pred_labels,
            probabilities=probs
        )

        report_dict = classification_report(
            true_labels,
            pred_labels,
            target_names=LABELS,
            zero_division=0,
            output_dict=True
        )

        with open(self.file_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)

            writer.writerow([])
            writer.writerow([f"EPOCH {epoch}"])

            writer.writerow([
                "epoch",
                "train_loss",
                "val_loss",
                "test_loss",
                "val_f1_macro",
                "val_f1_micro",
                "test_f1_macro",
                "test_f1_micro",
                "val_pearson_mean",
                "test_pearson_mean"
            ])

            for i in range(len(epoch_list)):
                writer.writerow([
                    epoch_list[i],
                    train_loss_list[i],
                    val_loss_list[i],
                    test_loss_list[i],
                    val_f1_macro_list[i],
                    val_f1_micro_list[i],
                    test_f1_macro_list[i],
                    test_f1_micro_list[i],
                    val_pearson_mean_list[i],
                    test_pearson_mean_list[i]
                ])

            # -------------------------
            # FINAL TEST SCORES AFTER THIS EPOCH
            # -------------------------
            writer.writerow([])
            writer.writerow([f"FINAL TEST SCORES AFTER EPOCH {epoch}"])
            writer.writerow(["metric", "value"])
            writer.writerow(["test_loss", test_loss])
            writer.writerow(["test_f1_macro", test_f1_macro])
            writer.writerow(["test_f1_micro", test_f1_micro])
            writer.writerow(["test_pearson_mean", test_pearson_mean])

            # -------------------------
            # CLASSWISE RESULTS AFTER THIS EPOCH
            # -------------------------
            writer.writerow([])
            writer.writerow([f"CLASSWISE RESULTS AFTER EPOCH {epoch}"])
            writer.writerow([
                "class",
                "precision",
                "recall",
                "f1_score",
                "support"
            ])

            for class_name in LABELS:
                row = report_dict.get(class_name, {})

                writer.writerow([
                    class_name,
                    row.get("precision", ""),
                    row.get("recall", ""),
                    row.get("f1-score", ""),
                    row.get("support", "")
                ])

        print(
            f"\nEpoch {epoch} cumulative losses, final test scores, "
            f"classwise results, and sentence-wise test results saved."
        )

        self._inside_eval = False


# =========================================================
# 14. MODEL
# =========================================================
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
)


# =========================================================
# 15. TRAINING ARGUMENTS
# =========================================================
training_args = TrainingArguments(
    output_dir="/content/distilbert_output",

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,

    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    # Early stopping monitors validation loss
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED
)


# =========================================================
# 16. TRAINER
# =========================================================
callback = SaveEpochResultsCallback(
    file_path=LOG_FILE,
    test_dataset=test_single,
    test_texts=test_texts,
    sentence_log_path=TEST_EPOCH_SENTENCE_LOG
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_single,
    eval_dataset=val_single,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[
        callback,
        EarlyStoppingCallback(
            early_stopping_patience=1,
            early_stopping_threshold=0.0
        )
    ]
)

callback.trainer_ref = trainer


# =========================================================
# 17. TRAIN
# =========================================================
start = time.time()

trainer.train()

end = time.time()

print(f"\nTotal training time: {end - start:.1f} seconds")
print("Epochwise result file saved at:", LOG_FILE)
print("Test sentence-wise epoch log saved at:", TEST_EPOCH_SENTENCE_LOG)


# =========================================================
# 18. FINAL TEST EVALUATION
# =========================================================
final_test_results = trainer.evaluate(
    eval_dataset=test_single,
    metric_key_prefix="final_test"
)

print("\nFinal test results:")
print(final_test_results)


# =========================================================
# 19. DISPLAY SAMPLE TEST SENTENCE LOG
# =========================================================
test_epoch_sentence_log_df = pd.read_csv(TEST_EPOCH_SENTENCE_LOG)

print("\nSample test sentence-wise log for every epoch:")
display(test_epoch_sentence_log_df.head(10))